In [55]:
import string
import tensorflow as tf
import os
import glob
import numpy as np

In [3]:
vocab = string.ascii_lowercase + " "
vocab = list(vocab)

In [7]:
char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="")
num_to_char = tf.keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)

In [83]:
def load_alignment(path : str):
    # path = bytes.decode(path.numpy())
    with open(path, "r") as f:
        lines = f.readlines()
    tokens = []
    for line in lines:
        start, end, text = line.split()
        if text!='sil':
            tokens.append(text)
    chars = list(" ".join(tokens))
    return char_to_num(chars)

In [97]:
def load_data(path : str):
    path = bytes.decode(path.numpy())
    speaker_name = path.split("\\")[-2]
    video_name = path.split("\\")[-1].split(".")[0]

    frames_path = os.path.join("..\\processed_data\\", f'{speaker_name}', f'{video_name}.npy')
    align_path  = os.path.join("..\\data\\", f'{speaker_name}', 'align',f'{video_name}.align')

    frames = np.load(frames_path)
    alignments = load_alignment(align_path)
    return frames, alignments

In [99]:
def mappable_function(path: str):
    try:
        result = tf.py_function(load_data, [path], (tf.float32, tf.int64))
    except Exception as e:
        print(f"Error in {path}: {e}")
        result = None
    return result

In [100]:
from sklearn.model_selection import train_test_split

In [101]:
all_videos = glob.glob("..\\processed_data\\s2_processed\\*.npy")

In [102]:
train, test = train_test_split(all_videos, test_size=0.2, random_state=42)

In [103]:
test

['..\\processed_data\\s2_processed\\lwbszs.npy',
 '..\\processed_data\\s2_processed\\sbwb4a.npy',
 '..\\processed_data\\s2_processed\\bwbt6s.npy',
 '..\\processed_data\\s2_processed\\lgal8s.npy',
 '..\\processed_data\\s2_processed\\pwwy2s.npy',
 '..\\processed_data\\s2_processed\\pgbrzs.npy',
 '..\\processed_data\\s2_processed\\srit8s.npy',
 '..\\processed_data\\s2_processed\\pbwp6s.npy',
 '..\\processed_data\\s2_processed\\lwar7p.npy',
 '..\\processed_data\\s2_processed\\prwx7p.npy',
 '..\\processed_data\\s2_processed\\swab5n.npy',
 '..\\processed_data\\s2_processed\\prbj3n.npy',
 '..\\processed_data\\s2_processed\\sgwqza.npy',
 '..\\processed_data\\s2_processed\\pgwk9n.npy',
 '..\\processed_data\\s2_processed\\srbb6a.npy',
 '..\\processed_data\\s2_processed\\bgan6a.npy',
 '..\\processed_data\\s2_processed\\sgbc6s.npy',
 '..\\processed_data\\s2_processed\\lwir3p.npy',
 '..\\processed_data\\s2_processed\\lrbk9n.npy',
 '..\\processed_data\\s2_processed\\bwbt5n.npy',
 '..\\processed_data

In [104]:
data = tf.data.Dataset.from_tensor_slices(train)
data = data.shuffle(500)
data = data.map(mappable_function, num_parallel_calls=tf.data.AUTOTUNE)
data = data.padded_batch(10, padded_shapes=([75, 48, 96, 1], [40]))
data = data.prefetch(tf.data.AUTOTUNE).cache()

val = tf.data.Dataset.from_tensor_slices(test)
val = val.shuffle(500)
val = val.map(mappable_function, num_parallel_calls=tf.data.AUTOTUNE)
val = val.padded_batch(10, padded_shapes=([75,  48, 96, 1], [40]))
val = val.prefetch(tf.data.AUTOTUNE).cache()

In [105]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv3D, Dense, LSTM, Bidirectional, Dropout, 
                                     MaxPool3D, Activation, Reshape, SpatialDropout3D, 
                                     BatchNormalization, TimeDistributed, Flatten, Input)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint
import tensorflow as tf
from tensorflow.keras import layers, models

In [106]:
def CTCLoss(y_true, y_pred):
    batch_size = tf.cast(tf.shape(y_true)[0], tf.int64)
    input_len = tf.cast(tf.shape(y_pred)[1], tf.int64)
    label_len = tf.cast(tf.shape(y_true)[1], tf.int64)
    
    input_len = input_len * tf.ones(shape=(batch_size, 1), dtype = tf.int64)
    label_len = label_len * tf.ones(shape=(batch_size, 1), dtype = tf.int64)
    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_len, label_len) 
    # deprecated, need to find the new method now
    return loss

In [116]:
model = Sequential()
model.add(Conv3D(128, 3, input_shape=(75,48,96,1), padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(Conv3D(256, 3, padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(Conv3D(75, 3, padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(TimeDistributed(Reshape((-1,))))  # flatten manually


model.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
model.add(Dropout(.5))

model.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
model.add(Dropout(.5))

model.add(Dense(char_to_num.vocabulary_size()+1, kernel_initializer='he_normal', activation='softmax'))

In [117]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv3d_18 (Conv3D)                   │ (None, 75, 48, 96, 128)     │           3,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_18 (Activation)           │ (None, 75, 48, 96, 128)     │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_18 (MaxPooling3D)      │ (None, 75, 24, 48, 128)     │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_19 (Conv3D)                   │ (None, 75, 24, 48, 256)     │         884,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_19 (Activation)           │ (None, 75, 24, 48, 256)     │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_19 (MaxPooling3D)      │ (None, 75, 12, 24, 256)     │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_20 (Conv3D)                   │ (None, 75, 12, 24, 75)      │         518,475 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_20 (Activation)           │ (None, 75, 12, 24, 75)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_20 (MaxPooling3D)      │ (None, 75, 6, 12, 75)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_6 (TimeDistributed) │ (None, 75, 5400)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_12 (Bidirectional)     │ (None, 75, 256)             │       5,661,696 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 75, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_13 (Bidirectional)     │ (None, 75, 256)             │         394,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 75, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 75, 29)              │           7,453 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,470,440 (28.50 MB)

 Trainable params: 7,470,440 (28.50 MB)

 Non-trainable params: 0 (0.00 B)

In [118]:
sample = val.as_numpy_iterator().next()

In [120]:
yhat = model.predict(sample[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step


In [123]:
def scheduler(epoch, lr):
    if epoch < 30:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

In [124]:
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

In [126]:
class ProduceExample(tf.keras.callbacks.Callback):
    def __init__(self, dataset) -> None:
        self.dataset = dataset.as_numpy_iterator()

    def on_epoch_end(self, epoch, logs=None) -> None:
        data = self.dataset.next()
        yhat = self.model.predict(np.array([data[0][0]]))
        decoded = tf.keras.backend.ctc_decode(yhat, [75], greedy=False)[0][0].numpy()

        print('Original:', tf.strings.reduce_join(num_to_char(data[1][0])).numpy().decode('utf-8'))
        print('Prediction:', tf.strings.reduce_join(num_to_char(decoded[0])).numpy().decode('utf-8'))


In [127]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)


In [128]:
checkpoint_callback = ModelCheckpoint(os.path.join('models','checkpoint.weights.h5'), monitor='loss', save_weights_only=True, save_best_only=True)


In [129]:
schedule_callback = LearningRateScheduler(scheduler)


In [131]:
example_callback = ProduceExample(val)


In [134]:
model.fit(data, validation_data=val, epochs=20, callbacks=[checkpoint_callback, schedule_callback, example_callback])

Epoch 1/20

 4/80 ━━━━━━━━━━━━━━━━━━━━ 42:32 34s/step - loss: 189.0652

KeyboardInterrupt: 